# Numerical check of analytic Jacobian determinant using JAX

- Based on [jnkepler](https://github.com/kemasuda/jnkepler) version 0.2.4
- Jacobian functions are implemented in [jnkepler.keplerian.jacobian](https://github.com/kemasuda/jnkepler/blob/main/src/jnkepler/keplerian/jacobian.py)

In [1]:
import os
os.environ["XLA_FLAGS"] = "--xla_cpu_use_thunk_runtime=false"
import jax.numpy as jnp
from jax import config
config.update('jax_enable_x64', True)

In [2]:
from jnkepler.keplerian.jacobian import *
from jnkepler.jaxttv.conversion import G 
import jnkepler
print("jnkepler version:", jnkepler.__version__)

jnkepler version: 0.2.4


## Full 3D Kepler Problem (Section 2.1, 2.1.1)

In [4]:
# test function
key_sets = {
    "am":   ("a", "ecc", "inc", "lnode", "omega", "M"),
    "atau": ("a", "ecc", "inc", "lnode", "omega", "tau"),
    "Pm":   ("period", "ecc", "inc", "lnode", "omega", "M"),
    "Ptau": ("period", "ecc", "inc", "lnode", "omega", "tau"),
}

def analytic_det(params, keys):
    ks = tuple(keys)
    if ks == key_sets["am"]:
        # needs 'a' and the implied 'period' (period can be computed inside numeric path;
        # for analytic path, 'period' is not required)
        return det_jkep_am(params)
    if ks == key_sets["atau"]:
        # needs period for the factor n = 2π/P
        if 'period' not in params:
            # if we have a, compute P for the analytic formula
            P = 2 * jnp.pi * jnp.sqrt(params['a']**3 / (G * params['mass']))
            params = dict(params, period=P)
        return det_jkep_atau(params)
    if ks == key_sets["Pm"]:
        return det_jkep_pm(params)
    if ks == key_sets["Ptau"]:
        return det_jkep_ptau(params)

In [6]:
# test parameter set
t_ref = 0.

params = dict(
    period=100.,
    ecc=0.2,
    inc=0.3,
    omega=0.5,
    lnode=1.0,
    tau=10.0,
    mass=1.5,
    t_ref=0.,
)

params['a'] = (G * params['mass'] / (2 * jnp.pi / params['period'])**2)**(1./3.)
params['M'] = 2 * jnp.pi * (t_ref - params['tau']) / params['period']
params['n'] = 2 * jnp.pi / params['period']

### sign and log|det|

In [7]:
for name, keys in key_sets.items():

    # JAX autodiff
    sign_num, logabs_num = slogdet_jkep_jax(params, keys)

    # Analytic
    det_ana = analytic_det(params, keys)
    sign_ana = jnp.sign(det_ana)
    logabs_ana = jnp.log(jnp.abs(det_ana) + 1e-300)

    print(f"# {keys}")
    print("  sign(det):     JAX: ", float(sign_num), "  analytic: ", float(sign_ana))
    print("  log|det|:      JAX: ", float(logabs_num), "    analytic: ", float(logabs_ana))
    print()

# ('a', 'ecc', 'inc', 'lnode', 'omega', 'M')
  sign(det):     JAX:  1.0   analytic:  1.0
  log|det|:      JAX:  -15.46580983827569     analytic:  -15.465809838275689

# ('a', 'ecc', 'inc', 'lnode', 'omega', 'tau')
  sign(det):     JAX:  -1.0   analytic:  -1.0
  log|det|:      JAX:  -18.233102957854438     analytic:  -18.233102957854435

# ('period', 'ecc', 'inc', 'lnode', 'omega', 'M')
  sign(det):     JAX:  1.0   analytic:  1.0
  log|det|:      JAX:  -21.204910597037582     analytic:  -21.204910597037582

# ('period', 'ecc', 'inc', 'lnode', 'omega', 'tau')
  sign(det):     JAX:  -1.0   analytic:  -1.0
  log|det|:      JAX:  -23.972203716616328     analytic:  -23.972203716616328



In [8]:
# speed
res = slogdet_jkep_jax(params, keys)
res.sign.block_until_ready()
res.logabsdet.block_until_ready()
_ = det_jkep_pm(params).block_until_ready()

print("JAX autodiff:")
%timeit res = slogdet_jkep_jax(params, keys); res.sign.block_until_ready(); res.logabsdet.block_until_ready()
print()
print("Analytic:")
%timeit out = det_jkep_am(params); out.block_until_ready()

JAX autodiff:
84.3 μs ± 536 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)

Analytic:
19.8 μs ± 743 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


## Reduced Jacobian fixing y (Section 2.1.2)

Here we expect
$$
\det J_\mathrm{kep} = x \det J_\mathrm{kep}',
$$
where $x$ and $\det J_\mathrm{kep}' = \det\left(\partial(x,z,v_x,v_y,v_z) \over \partial(a,e,i,\omega,M)\right)$ in the right-hand side is evaluated at a given $y$.

In [9]:
from jnkepler.keplerian import *
from jax import jit, jacrev
from functools import partial

@partial(jit, static_argnums=(1,))
def slogdet_jkep5_jax(params, keys, y_value):
    """Comouting Jacobian determinant using JAX autodiff for given y_value"""

    def lnode_given_y(params, y_value):
        '''fix \Omega implicitly from given y'''
        x = elements_to_xv(0., params | {'lnode': 0.})['x'][0]
        x0, y0 = x[0], x[1]
        r0, theta0 = jnp.sqrt(x0**2 + y0**2), jnp.arctan2(y0, x0)
        thetaomega = jnp.arcsin(y_value / r0) # choose the (-pi/2, pi/2) branch
        lnode = thetaomega - theta0
        return lnode, r0 * jnp.cos(thetaomega) # \Omega*, x given y

    def func(params):
        if 'a' in keys:
            params['period'] = 2 * jnp.pi * jnp.sqrt(params['a']**3 / G / params['mass'])
        elif 'n' in keys:
            params['period'] = 2 * jnp.pi / params['n']
        if 'M' in keys:
            params['tau'] = params['t_ref'] - params['M'] * params['period'] / 2 / jnp.pi

        params_ = params | {'lnode': lnode_given_y(params, y_value)[0]}
        out = elements_to_xv(0., params_)
        return jnp.hstack([out['x'][0][0], out['x'][0][2], out['v'][0]])
    
    params_ = params.copy()
    del params_['lnode']
    Jdict = jacrev(func)(params_)
    Jarr = jnp.stack([Jdict[k].reshape(-1) for k in keys], axis=1)

    _, x0 = lnode_given_y(params_, y_value)

    return jnp.linalg.slogdet(Jarr), x0

<>:10: SyntaxWarning: invalid escape sequence '\O'
<>:10: SyntaxWarning: invalid escape sequence '\O'
/var/folders/qp/91qlh0v11sb02z6vb4nb_s0m0000gn/T/ipykernel_39975/1875527013.py:10: SyntaxWarning: invalid escape sequence '\O'
  '''fix \Omega implicitly from given y'''


In [10]:
keys = ('a', 'ecc', 'inc', 'lnode', 'omega', 'M')
keys5 = ('a', 'ecc', 'inc', 'omega', 'M')

x0 = elements_to_xv(0., params)['x'][0]
max_y = jnp.sqrt(x0[0]**2 + x0[1]**2)

for y_value_frac in jnp.arange(-1., 1, 0.5):
    y_value = y_value_frac * max_y
    logJdet_sign, logJdet_val = slogdet_jkep_jax(params, keys)

    (logJdet5_sign, logJdet5_val), x = slogdet_jkep5_jax(params, keys5, y_value)
    print(f"# y={y_value}")
    print(f"log|detJkep| = {logJdet_val} (sign: {logJdet_sign})")
    print(f"log|detJkep'| = {logJdet5_val} (sign: {logJdet5_sign}), x={x}")
    print(f"log|x * detJkep'| = {jnp.log(x) + logJdet5_val}")
    print()

# y=-0.4101723190311828
log|detJkep| = -15.46580983827569 (sign: 1.0)
log|detJkep'| = 3.447194761392802 (sign: 1.0), x=6.112043868253773e-09
log|x * detJkep'| = -15.465809846317608

# y=-0.2050861595155914
log|detJkep| = -15.46580983827569 (sign: 1.0)
log|detJkep'| = -14.430790884788733 (sign: 1.0), x=0.35521964821017976
log|x * detJkep'| = -15.465809838275685

# y=0.0
log|detJkep| = -15.46580983827569 (sign: 1.0)
log|detJkep'| = -14.574631921014628 (sign: 1.0), x=0.41017231903118284
log|x * detJkep'| = -15.46580983827569

# y=0.2050861595155914
log|detJkep| = -15.46580983827569 (sign: 1.0)
log|detJkep'| = -14.430790884788737 (sign: 1.0), x=0.35521964821017976
log|x * detJkep'| = -15.465809838275689



## 2D Kepler Problem (Section 2.2)

In [11]:
@partial(jit, static_argnums=(1,))
def slogdet_jkep4_jax(params, keys):
    """Comouting Jacobian determinant using JAX autodiff"""
    def func(params):
        if 'a' in keys:
            params['period'] = 2 * jnp.pi * jnp.sqrt(params['a']**3 / G / params['mass'])
        elif 'n' in keys:
            params['period'] = 2 * jnp.pi / params['n']
        if 'M' in keys:
            params['tau'] = params['t_ref'] - params['M'] * params['period'] / 2 / jnp.pi
        out = elements_to_xv(0., params | {'lnode': 0, 'inc': jnp.pi / 2.})
        return jnp.hstack([out['x'][0][0], out['x'][0][2], out['v'][0][0], out['v'][0][2]])
    
    exclude = {'lnode', 'inc'}
    params4 = {k: v for k, v in params.items() if k not in exclude}
    Jdict = jacrev(func)(params4)
    Jarr = jnp.stack([Jdict[k].reshape(-1) for k in keys], axis=1)

    return jnp.linalg.slogdet(Jarr)


@jit
def det_jkep4_am(params):
    """Analytic Jacobian determinant for (a,e,i,Omega,omega,M)"""
    mu, e = G * params['mass'], params['ecc']
    det = 0.5 * mu * e / jnp.sqrt(1. - e**2)
    return det

In [12]:
keys4 = ('a', 'ecc', 'omega', 'M')
sign, logabsdet = slogdet_jkep4_jax(params, keys4)
det_analytic = det_jkep4_am(params)
print(f"JAX autodiff:    sign = {sign},  log|det| = {logabsdet}")
print(f"analytic:        sign = {jnp.sign(det_analytic)}, log|det| = {jnp.log(jnp.abs(det_analytic))}")

JAX autodiff:    sign = 1.0,  log|det| = -10.002156728888316
analytic:        sign = 1.0, log|det| = -10.002156728888316
